In [40]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("ANAC_mp_loc.csv")

#fatal - resulting in death of at least one member of the party
#serious - requiring emergency medical attention/rescue. Includes head injuries, fractures, spinal cord injury, HAPE/HACE
#moderate - requiring non-emergent medical attention. Includes lacerations, cuts, scrapes, sprains
#minor - no injury or injuries not requiring medical attention


SEVERITY_PATTERNS = {
    "fatal": [
        r"\bfatal injuries?\b",
        r"\bfatal\b",
        r"\bobituary\b",
        r"\bfatality\b",
        r"\bfatalities\b",
        r"\b(fatality)\b",
        r"\bdeceased\b",
        r"\bbody\b",
        r"\bbodies\b",
        r"\brecovery efforts\b",
        r"\bkilled\b",
        r"\bdied\b",
        r"\bdeaths?\b",
        r"\bdead\b",
        r"\bsuffocated\b",
        r"\bCPR\b",
        r"\bburied\b",
        r"\blost (?:his|her|their) life\b",
        r"\bdid not survive\b",
        r"\bsuccumbed\b",
        r"\b(?:the |his |her |their )?body was recovered\b",
        r"\b(?:the |his |her |their )?body (?:was )?recovered\b"
        r"\b(?:remains?|bodies?|body) .*?\b(?:recovered|discovered)\b"
    ],
    "serious": [
        r"\bunconscious\b",
        r"\bunresponsive\b",
        r"\bcrushed\b",
        r"\bsurgery\b",
        r"\bhyperbaric\b",
        r"\bpneumothorax\b",
        r"\btraumatic brain injury\b",
        r"\bhead injury\b",
        r"\bpelvic injury\b",
        r"\bback injury\b",
        r"\bvertebrae\b",
        r"\bvertebra\b",
        r"\btbi\b",
        r"\brespiratory distress\b",
        r"\bhape\b",
        r"\bhace\b",
        r"\bcerebral edema\b",
        r"\bpulmonary edema\b",
        r"\bappendicitis\b", 
        r"\bseizures?\b",
        r"\bamputation\b",
        r"\bamputated\b",
        r"\bhypothermia\b",
        r"\bfrostbitten\b",
        r"\bfrostbite\b",
        r"\btraumatic brain injury\b"
        r"\bloss of consciousness\b",
        r"\bconcussion\b",
        r"\bbleeding\b",
        r"\btourniquets?\b",
        r"\bhead lacerations?\b",
        r"\bfractures?\b",
        r"\bfractur(?:ed|ing)\b",
        r"\bmultiple fractures?\b",
        r"\bbroken\b",
        r"\bbroke\b",
        r"\bbreak\b",
        r"\bsplint(?:ed|ing)\b",
        r"\bsevere injuries?\b",
        r"\bsevere enough\b",
        r"\bpermanently damaged\b",
        r"\bseriously damaged\b",
        r"\blife-threatening\b",
        r"\bwould die\b",
        r"\bparalysis\b",
        r"\bbreak(?:ing|s|en)?\b",
        r"\b(?:lost|loss of) consci?ousness\b"
    ],
    "moderate": [
        r"\bburns?\b",
        r"\bbite(?:n|s)\b",
        r"\binjured\b",
        r"\bams\b",
        r"\baltitude illness\b",
        r"\bacute mountain sickness\b",
        r"\bsnow binlindness\b",
        r"\bwound\b",
        r"\bwounds\b",
        r"\blacerations?\b",
        r"\btenderness\b",
        r"\babrasions?\b",
        r"\bbruises?\b",
        r"\bcuts?\b",
        r"\bscrapes?\b",
        r"\bcontusions?\b",
        r"\bhematoma\b",
        r"\bsprain(?:ed|s)?\b",
        r"\bstrain(?:ed|s)?\b",
        r"\bdislocat(?:ed|ion|ions)\b",
        r"\bgash(?:es)?\b",
        r"\bcramping\b",
        r"\bdizziness\b",
        r"\bhypothermic\b",
        r"\bdamaged\b",
        r"\bswelling\b",
        r"\b(?:ankle|back|arm|leg|head|neck|shoulder|knee|wrist) injuries\b",
        r"\binjuries\b",
        r"\binjury\b",
        r"\binjured\b",
        
    ],
    "minor": [
        r"\bminor injuries?\b",
        r"\bexhausted\b",
        r"\bstranded\b",
        r"\bstuck\b",
        r"\bbenighted\b",
        r"\btrapped\b",
        r"\bunhurt\b",
        r"\buninjured\b",
        r"\bavulsion\b",
        r"\btwisted\b",
        r"\bpain\b",
        r"\bspasm\b",
        r"\bno injuries?\b",
        r"\bbruis(?:e|s|ed)\b",
        r"\bsore\b",
        r"\bscrapes?\b",
        r"\bno injury\b",
        r"\bonly abrasions?\b",
        r"\bonly bruises?\b",
        r"\bwalked away\b",
        r"\bself-?rescued\b",
        r"\bslight injuries?\b",
        r"\bdehydrat(?:ed|ion)\b"
    ],
}



  
ORDER = ["fatal", "serious", "moderate", "minor"]


def classify_severity_with_matches(text):
    if not isinstance(text, str) or not text.strip():
        return None, []

    t = text.lower()
    t = re.sub(r"\s+", " ", t)

    matches = []

    for label in ORDER:
        for pattern in SEVERITY_PATTERNS[label]:
            if re.search(pattern, t):
                matches.append((label, pattern))

    # determine final severity by priority
    final = None
    for label in ORDER:
        if any(m[0] == label for m in matches):
            final = label
            break

    return final, matches

df[["severity", "matched_terms"]] = df["Text"].apply(
    lambda x: pd.Series(classify_severity_with_matches(x))
)

print (df.head())


  ID                                     Accident Title  Publication Year  \
0  2  Failure of Rappel—Failure to Check System, Bri...            1990.0   
1  3  Fall into Crevasse, Climbing Alone, Inadequate...            1990.0   
2  4  Fall into Crevasse, Climbing Unroped, British ...            1990.0   
3  5  Fall Into Crevasse, Unroped, Inadequate Equipm...            1990.0   
4  6  Fall into Moat, Descending Unroped, Poor Posit...            1990.0   

                                                Text  \
0  British Columbia, Squamish, Smoke Bluffs\nOn M...   
1  Alberta, Rocky Mountains, Crowfoot Mountain\nO...   
2  British Columbia, Bugaboo Mountains, Bugaboo S...   
3  On the afternoon of March 29, 1989, four ski t...   
4  Alberta, Rocky Mountains, Mount Nublock\nLate ...   

                                        Tags Applied Country  \
0  Experienced, Serious, Descent, Roped, Top-Rope...      CA   
1  Experienced, Minor, Unroped , Solo, Climbing A...      CA   
2  Minor

In [41]:
df.to_csv('ANAC_mp_loc_severity', index=False) 

In [42]:
value_counts = df['severity'].value_counts()

In [39]:
print (value_counts)

severity
fatal       1202
serious     1023
minor        234
moderate     187
Name: count, dtype: int64
